# Gaussian-state projection onto PIAB eigenstates

**Prerequisite:** [Sampled wavefunction normalization and basis projection](../../../docs/lecture-notes/qm/piab1d/piab1d__basis_projection.md)

This maintained laboratory extracts the Gaussian construction, weighted normalization, and energy-eigenbasis projection behind the historical PIAB1D expansion-probability visualization. The selected model is nondimensional: $L=1$, $m=1$, and $\hbar=1$.

## Learning objectives

1. Construct and normalize a sampled Gaussian state under explicit quadrature weights.
2. Project the state onto analytical PIAB eigenfunctions.
3. Interpret squared expansion coefficients as represented probabilities.
4. Quantify probability capture and reconstruction error for a finite basis.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab1d.base import Piab1D
from projectkoios.physkit.qm.piab1d.tise_analytical.solution import (
    Piab1DAnalyticalSolution,
)
from projectkoios.physkit.qm.sampled_states import (
    GaussianWavePacket1DInitialState,
    GaussianWavePacket1DInitialStateModel,
)
from projectkoios.physkit.units import (
    ScalarQuantity,
    Unitless,
    UnitSystem,
    VectorQuantity,
)

In [ ]:
model = Piab1D(length=1.0, mass=1.0, unit_system=UnitSystem.NONDIMENSIONAL)
interior_point_count = 400
mode_count = 60
grid_spacing = model.length / (interior_point_count + 1)
coordinate_values = np.linspace(
    grid_spacing,
    model.length - grid_spacing,
    interior_point_count,
    dtype=np.float64,
)
coordinates = VectorQuantity(magnitude=coordinate_values, unit=Unitless())
quadrature_weights = VectorQuantity(
    magnitude=np.full(interior_point_count, grid_spacing, dtype=np.float64),
    unit=Unitless(),
)

packet_center = model.length / 2.0
packet_width = model.length / 10.0
zero_wavenumber = 0.0
initial_state = GaussianWavePacket1DInitialState(
    model=GaussianWavePacket1DInitialStateModel(
        center=ScalarQuantity(magnitude=packet_center, unit=Unitless()),
        width=ScalarQuantity(magnitude=packet_width, unit=Unitless()),
        wavenumber=ScalarQuantity(magnitude=zero_wavenumber, unit=Unitless()),
    )
)
sampled_initial_state = initial_state.evaluate(coordinates=coordinates)
normalization = sampled_initial_state.normalize(quadrature_weights=quadrature_weights)

In [ ]:
stationary_states = Piab1DAnalyticalSolution(model=model).evaluate(count=mode_count)
eigenfunctions = stationary_states.eigenfunctions(coordinates=coordinates)
projection = eigenfunctions.project(
    state=normalization.normalized_state,
    quadrature_weights=quadrature_weights,
)
reconstructed_state = (
    eigenfunctions.amplitude_matrix.magnitude @ projection.coefficients.magnitude
)
reconstruction_error = float(
    np.sqrt(
        np.sum(
            np.abs(
                reconstructed_state
                - normalization.normalized_state.amplitudes.magnitude
            )
            ** 2
            * quadrature_weights.magnitude
        )
    )
)

probability_tolerance = 1.0e-12
reconstruction_tolerance = 1.0e-6
forbidden_mode_tolerance = 1.0e-28
assert projection.represented_probability > 1.0 - probability_tolerance
assert reconstruction_error < reconstruction_tolerance
assert np.all(projection.probabilities.magnitude[1:20:2] < forbidden_mode_tolerance)
projection.represented_probability, reconstruction_error

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(
    coordinate_values,
    np.abs(normalization.normalized_state.amplitudes.magnitude) ** 2,
    label="sampled Gaussian",
)
axes[0].plot(
    coordinate_values,
    np.abs(reconstructed_state) ** 2,
    linestyle="--",
    label=f"{mode_count}-mode reconstruction",
)
axes[0].set(
    xlabel="$x/L$",
    ylabel=r"$|\psi(x)|^2$",
    title="Normalized state and finite-basis reconstruction",
)
axes[0].grid(True)
axes[0].legend()

shown_modes = 15
axes[1].bar(
    stationary_states.quantum_numbers[:shown_modes],
    projection.probabilities.magnitude[:shown_modes],
)
axes[1].set(
    xlabel="quantum number $n$",
    ylabel=r"$|c_n|^2$",
    title="Expansion probabilities",
    xticks=stationary_states.quantum_numbers[:shown_modes],
)
axes[1].grid(True, axis="y")

figure.tight_layout()
plt.show()

## Interpretation

The Gaussian is centered at the reflection-symmetric midpoint and has zero average wavenumber. Consequently, only PIAB eigenfunctions with the same reflection parity contribute; the alternating coefficients are zero to numerical precision. The first, third, and fifth modes dominate the expansion.

The represented probability is the sum of $|c_n|^2$ over the selected 60-mode inventory. The nonzero reconstruction error is a finite-basis effect and also reflects the small mismatch between a free Gaussian tail and exact homogeneous-Dirichlet boundary behavior.

## Exercises

1. Move the Gaussian center away from $x/L=0.5$ and examine the parity selection rule.
2. Set a nonzero wavenumber and inspect the complex expansion coefficients.
3. Repeat with 10, 20, 40, and 80 modes and plot reconstruction error against mode count.
4. Replace the rectangular interior-grid weights with a declared alternative quadrature representation and compare results.
